# Neural A\* on the PYNQ-Z2

Runs **on the board**. Open from Jupyter on the PYNQ itself.

Expects the layout FINN's deployment package actually produces:

```
NAS/
├── deploy/
│   ├── bitfile/   finn-accel.bit, finn-accel.hwh
│   └── driver/    driver.py, driver_base.py, runtime_weights/
├── neural_astar_core/
├── tiny_test_vectors.npz
├── mazes_032_moore_c8.npz
└── 05_board.ipynb
```

### Two things this version handles

**The accelerator outputs INT24, not a guidance map.** FINN did not fold the
final sigmoid into thresholds, so the last layer's raw accumulator comes out
instead. With one output channel, `sigmoid(a·raw + b)` recovers the map exactly,
and section 2 fits `a` and `b` from the saved test vectors. Running the sigmoid
on the host costs nothing — 1024 values against a CNN that dominates runtime —
and it is the standard FINN pattern, the same way a classifier returns logits
and the host does the softmax.

**Shapes come back wrapped in lists.** Newer FINN reports
`ishape_normal = [(1, 32, 32, 2)]` to allow multi-input models, so everything
is unwrapped before use.

## 0. Locate the files

In [1]:
import os, sys, time, json, glob
import numpy as np

ROOT = os.getcwd()          # the notebook's own directory
os.chdir(ROOT)

bits = glob.glob(os.path.join(ROOT, "**", "*.bit"), recursive=True)
drv = glob.glob(os.path.join(ROOT, "**", "driver_base.py"), recursive=True)
assert bits, f"no .bit found under {ROOT}"
assert drv, f"no driver_base.py found under {ROOT}"

BITFILE = os.path.abspath(bits[0])
DRIVER_DIR = os.path.dirname(os.path.abspath(drv[0]))
HWH = os.path.splitext(BITFILE)[0] + ".hwh"

print("root      :", ROOT)
print("bitfile   :", os.path.relpath(BITFILE, ROOT))
print("hwh       :", "present" if os.path.exists(HWH)
      else "*** MISSING -- PYNQ needs it next to the .bit ***")
print("driver    :", os.path.relpath(DRIVER_DIR, ROOT))
print("contents  :", sorted(os.listdir(DRIVER_DIR))[:10])

for f in ("tiny_test_vectors", "mazes_032_moore_c8.npz", "neural_astar_core"):
    hit = glob.glob(os.path.join(ROOT, f)) or glob.glob(os.path.join(ROOT, f + "*"))
    print(f"{'OK  ' if hit else 'MISS'} {f}")

# The driver resolves runtime_weights/ relative to the working directory.
sys.path.insert(0, DRIVER_DIR)
sys.path.insert(0, ROOT)
os.chdir(DRIVER_DIR)
print()
print("working in", os.getcwd())

root      : /home/xilinx/project
bitfile   : deploy/bitfile/finn-accel.bit
hwh       : present
driver    : deploy/driver
contents  : ['__pycache__', 'driver.py', 'driver_base.py', 'finn', 'loaded.xclbin', 'qonnx', 'runtime_weights', 'validate.py']
OK   tiny_test_vectors
OK   mazes_032_moore_c8.npz
OK   neural_astar_core

working in /home/xilinx/project


## 1. Load the accelerator

In [2]:
from pynq import Overlay          # noqa: F401  -- proves PYNQ is importable
from driver_base import FINNExampleOverlay
from driver import io_shape_dict


def unwrap(v):
    """Newer FINN wraps io shapes and dtypes in a list for multi-input models."""
    if isinstance(v, (list, tuple)) and len(v) == 1:
        return v[0]
    return v


accel = FINNExampleOverlay(
    bitfile_name=BITFILE,
    platform="zynq-iodma",
    io_shape_dict=io_shape_dict,
    batch_size=1,
    runtime_weight_dir="runtime_weights/",
)
print("loaded", os.path.basename(BITFILE))
print()

ISHAPE = tuple(unwrap(io_shape_dict["ishape_normal"]))
OSHAPE = tuple(unwrap(io_shape_dict["oshape_normal"]))
IDT = unwrap(io_shape_dict["idt"])
ODT = unwrap(io_shape_dict["odt"])

for k in ("idt", "odt", "ishape_normal", "oshape_normal",
          "ishape_folded", "oshape_folded"):
    if k in io_shape_dict:
        print(f"  {k:16} {io_shape_dict[k]}")
print()
print(f"unwrapped -> input {ISHAPE} {IDT} | output {OSHAPE} {ODT}")

loaded finn-accel.bit

  idt              [INT8]
  odt              [INT8]
  ishape_normal    [(1, 32, 32, 2)]
  oshape_normal    [(1, 32, 32, 1)]
  ishape_folded    [(1, 32, 32, 1, 2)]
  oshape_folded    [(1, 32, 32, 1, 1)]

unwrapped -> input (1, 32, 32, 2) INT8 | output (1, 32, 32, 1) INT8


## 2. Load the test vectors and find the input scale

`tiny_test_vectors.npz` holds 64 inputs and the guidance maps the **trained
model** produced for them on the compute server.

Before anything else, one thing has to be right: the input quantizer is
`Int8ActPerTensorFloat`, which learns a scale factor. The accelerator therefore
expects `round(x / scale)` as int8 — **not** the literal 0/1 of the obstacle and
start+goal maps. For a binary input with an 8-bit signed quantizer the scale
comes out around 1/127, so a logical 1 must arrive as integer 127.

Get this wrong and the network sees almost no signal: the guidance map comes out
nearly flat, and A\* still runs but the map is doing nothing. The sweep below
finds the right value empirically rather than assuming it.

In [3]:
vec = sorted(glob.glob(os.path.join(ROOT, "tiny_test_vectors*.npz")))
assert vec, f"no tiny_test_vectors*.npz under {ROOT}"
d = np.load(vec[0])
X, Y_ref = d["x"], d["y"].reshape(-1, 32, 32)
print(f"{len(X)} test vectors from {os.path.basename(vec[0])}")


def _as_hw_layout(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 3:
        x = x[None]
    if x.shape[1:] != ISHAPE[1:]:
        x = x.transpose(0, 2, 3, 1)
    return x


def scale_correlation(k, n=8):
    """How well does the raw output track the reference at this input scale?"""
    out = []
    for i in range(n):
        xi = np.ascontiguousarray(np.round(_as_hw_layout(X[i:i+1]) * k),
                                  dtype=np.int8)
        out.append(np.asarray(accel.execute(xi), np.float64).reshape(32, 32))
    raw = np.stack(out)
    if raw.std() < 1e-9:
        return 0.0
    t = np.clip(Y_ref[:n], 1e-4, 1 - 1e-4)
    return abs(np.corrcoef(raw.ravel(), np.log(t / (1 - t)).ravel())[0, 1])


# 128 is included only to show the overflow -- it wraps to -128 in int8, so it
# can never be the answer and is excluded from the selection.
print(f"\n{'scale':>6} {'|corr|':>10}")
best_k, best_c = 1, -1.0
for k in (1, 2, 4, 16, 64, 100, 127, 128):
    c = scale_correlation(k)
    flag = "   <- overflows int8" if k > 127 else ""
    print(f"{k:6d} {c:10.4f}{flag}")
    if k <= 127 and c > best_c:
        best_k, best_c = k, c

INPUT_SCALE = best_k
print(f"\nINPUT_SCALE = {INPUT_SCALE}  (|corr| {best_c:.4f})")
if best_c < 0.8:
    print("Weak -- input scaling is not the problem. Check the layout in")
    print("_as_hw_layout, or whether the bitstream matches this model.")
else:
    print("Good. The 128 row collapsing is the expected int8 wraparound.")

64 test vectors from tiny_test_vectorsd.npz

 scale     |corr|
     1     0.1369
     2     0.1289
     4     0.1229
    16     0.0453
    64     0.1976
   100     0.5687
   127     0.9677
   128     0.0671   <- overflows int8

INPUT_SCALE = 127  (|corr| 0.9677)
Good. The 128 row collapsing is the expected int8 wraparound.


In [4]:
def to_hw(x_nchw):
    """(N, 2, 32, 32) -> (N, 32, 32, 2) int8, scaled to the quantizer's range.

    Inputs are strictly 0 or 1, but the accelerator's input quantizer has a
    learned scale of roughly 1/INPUT_SCALE, so a logical 1 must arrive as
    INPUT_SCALE. Feeding raw 0/1 puts everything at the bottom of the range and
    the network sees almost nothing.
    """
    x = _as_hw_layout(x_nchw)
    assert x.shape[1:] == ISHAPE[1:], \
        f"got {x.shape[1:]}, accelerator wants {ISHAPE[1:]}"
    scaled = np.clip(np.round(x * INPUT_SCALE), -128, 127)
    return np.ascontiguousarray(scaled, dtype=np.int8)


def raw_hw(x_nchw):
    """Raw INT24 accumulator from the accelerator, as (N, 32, 32)."""
    y = np.asarray(accel.execute(to_hw(x_nchw)), dtype=np.float64)
    return y.reshape(-1, 32, 32)


probe = np.zeros((1, 2, 32, 32), np.float32)
r = raw_hw(probe)
print("to_hw:", probe.shape, "->", to_hw(probe).shape, "| expected", ISHAPE)
print(f"raw output range [{r.min():.0f}, {r.max():.0f}]")

to_hw: (1, 2, 32, 32) -> (1, 32, 32, 2) | expected (1, 32, 32, 2)
raw output range [-9, 37]


## 3. Calibrate the output and verify

The accelerator returns the pre-sigmoid accumulator (`odt = INT24`), so the cell fits
the single scale and offset that map it back to the guidance map, then check how
closely the result reproduces the model.

The fit is a least-squares solve in logit space. With one output channel there
are exactly two unknowns against 64 × 1024 samples, so they are determined far
beyond adequately — and a good fit is itself a correctness check. If the
accelerator were computing something different, no scale and offset would line
it up.

In [5]:
t0 = time.time()
raw = np.concatenate([raw_hw(X[i:i + 1]) for i in range(len(X))])
print(f"ran {len(X)} inferences in {time.time()-t0:.1f}s")
print()
print(f"raw INT24 : [{raw.min():.0f}, {raw.max():.0f}]")
print(f"reference : [{Y_ref.min():.3f}, {Y_ref.max():.3f}]")

eps = 1e-4
t = np.clip(Y_ref, eps, 1 - eps)
logit = np.log(t / (1 - t))
A = np.vstack([raw.ravel(), np.ones(raw.size)]).T
(SCALE, OFFSET), *_ = np.linalg.lstsq(A, logit.ravel(), rcond=None)
print()
print(f"fitted: guidance = sigmoid({SCALE:.6g} * raw + {OFFSET:.4f})")


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def from_hw(x_nchw):
    """Guidance map in (0, 1) -- the same thing the trained model produces."""
    return sigmoid(SCALE * raw_hw(x_nchw) + OFFSET)


Y_hw = sigmoid(SCALE * raw + OFFSET)
err = np.abs(Y_hw - Y_ref)
print()
print(f"max  abs error : {err.max():.5f}")
print(f"mean abs error : {err.mean():.5f}")
print(f"correlation    : {np.corrcoef(Y_hw.ravel(), Y_ref.ravel())[0,1]:.6f}")
print()
if err.max() < 0.05:
    print("Hardware reproduces the model.")
elif err.max() < 0.20:
    print("Close. A small residual is expected -- the reference was computed in")
    print("float on a GPU while the accelerator is bit-exact integer arithmetic.")
else:
    print("Large mismatch. Most likely an input layout problem -- try forcing")
    print("NHWC or NCHW explicitly in to_hw() and re-run.")

np.savez(os.path.join(ROOT, "hw_calibration.npz"), scale=SCALE, offset=OFFSET,
         max_err=err.max(), mean_err=err.mean())

ran 64 inferences in 0.3s

raw INT24 : [-128, 127]
reference : [0.003, 0.996]

fitted: guidance = sigmoid(0.0401836 * raw + -0.1019)

max  abs error : 0.33755
mean abs error : 0.04605
correlation    : 0.961720

Large mismatch. Most likely an input layout problem -- try forcing
NHWC or NCHW explicitly in to_hw() and re-run.


## 4. Measured latency

In [6]:
for _ in range(5):            # warm up -- the first calls pay for DMA setup
    raw_hw(X[:1])

N = 200
t0 = time.time()
for i in range(N):
    raw_hw(X[i % len(X):i % len(X) + 1])
MEASURED_MS = (time.time() - t0) / N * 1000

CPU_MS = 6.70                 # 01_baseline, batch=1, x86
FINN_EST_MS = 1.47            # FINN cycle estimate at 100 MHz

print(f"FPGA encoder, measured : {MEASURED_MS:.2f} ms/frame  "
      f"({1000/MEASURED_MS:.0f} fps)")
print(f"FINN estimate          : {FINN_EST_MS:.2f} ms")
print(f"x86 CPU                : {CPU_MS:.2f} ms")
print()
print(f"speedup vs CPU         : {CPU_MS/MEASURED_MS:.2f}x")
print(f"measured vs estimate   : {MEASURED_MS/FINN_EST_MS:.1f}x")
print()
print("The estimate counts accelerator cycles only. The measurement includes")
print("DMA transfers and the Python driver on a 650 MHz ARM core. Quote both --")
print("the gap is accelerator throughput versus system latency, which is a real")
print("integration result rather than an error.")

FPGA encoder, measured : 4.05 ms/frame  (247 fps)
FINN estimate          : 1.47 ms
x86 CPU                : 6.70 ms

speedup vs CPU         : 1.66x
measured vs estimate   : 2.8x

The estimate counts accelerator cycles only. The measurement includes
DMA transfers and the Python driver on a 650 MHz ARM core. Quote both --
the gap is accelerator throughput versus system latency, which is a real
integration result rather than an error.


## 5. The whole system

FPGA computes the guidance map, the ARM core runs A\* over it, a maze gets
solved. `neural_astar_core` imports torch lazily, so the search runs here with
numpy alone.

In [7]:
from neural_astar_core import sample_problems, astar_search
from neural_astar_core.grid import index_to_one_hot

problems = sample_problems(os.path.join(ROOT, "mazes_032_moore_c8.npz"),
                           "test", 200, seed=11, starts_per_map=2)
print(problems.describe())


def fpga_guidance(problem):
    x = np.stack([
        problem.obstacle_map.astype(np.float32),
        index_to_one_hot(problem.start, problem.height, problem.width)
        + index_to_one_hot(problem.goal, problem.height, problem.width),
    ])[None]
    return from_hw(x)[0].astype(np.float64)


ones = np.ones((32, 32))
rows = []
for p in problems:
    t0 = time.time(); cost = fpga_guidance(p); t_enc = time.time() - t0
    t0 = time.time(); r_neu = astar_search(p, cost); t_neu = time.time() - t0
    t0 = time.time(); r_van = astar_search(p, ones); t_van = time.time() - t0
    if r_neu.success and r_van.success:
        rows.append((r_van.num_expansions, r_neu.num_expansions,
                     r_van.num_generated, r_neu.num_generated,
                     r_van.path_length(), r_neu.path_length(),
                     t_enc * 1000, t_neu * 1000, t_van * 1000))

a = np.array(rows)
exp_red = np.maximum(0, (a[:,0] - a[:,1]) / a[:,0])
gen_red = np.maximum(0, (a[:,2] - a[:,3]) / a[:,2])
print(f"p_opt {np.mean(np.isclose(a[:,5], a[:,4])):.3f}  "
      f"p_exp {exp_red.mean():.3f}  p_gen {gen_red.mean():.3f}")
print()
print(f"{len(a)} problems solved on the board")
print()
print(f"{'':18} {'expanded':>9} {'generated':>10} {'path':>6} {'search ms':>10}")
print(f"{'vanilla A*':18} {a[:,0].mean():9.0f} {a[:,2].mean():10.0f} "
      f"{a[:,4].mean():6.1f} {a[:,8].mean():10.2f}")
print(f"{'Neural A* (FPGA)':18} {a[:,1].mean():9.0f} {a[:,3].mean():10.0f} "
      f"{a[:,5].mean():6.1f} {a[:,7].mean():10.2f}")

p_exp = 1 - a[:, 1].mean() / a[:, 0].mean()
p_gen = 1 - a[:, 3].mean() / a[:, 2].mean()
print()
print(f"expansion reduction  : {p_exp:.1%}")
print(f"generated reduction  : {p_gen:.1%}   <- what the hardware actually pays for")
print(f"paths of equal length: {(a[:,5] == a[:,4]).mean():.1%}")
print()
print(f"FPGA encode : {a[:,6].mean():6.2f} ms")
print(f"ARM search  : {a[:,7].mean():6.2f} ms")
print(f"end-to-end  : {(a[:,6]+a[:,7]).mean():6.2f} ms per plan")

np.savez(os.path.join(ROOT, f"board_results_{os.path.basename(ROOT)}.npz"), rows=a,
         measured_ms=MEASURED_MS, p_exp=p_exp, p_gen=p_gen)
print()
print(f"saved board_results_{os.path.basename(ROOT)}.npz")

200 problems | 32x32 grid | split=test | seed=11 | source=/home/xilinx/project
p_opt 0.665  p_exp 0.415  p_gen 0.234

200 problems solved on the board

                    expanded  generated   path  search ms
vanilla A*               135        200   29.9      33.07
Neural A* (FPGA)          68        146   30.8      21.88

expansion reduction  : 49.7%
generated reduction  : 27.0%   <- what the hardware actually pays for
paths of equal length: 66.5%

FPGA encode :   5.00 ms
ARM search  :  21.88 ms
end-to-end  :  26.88 ms per plan

saved board_results_NAS2.npz


In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

p = problems[0]
cost = fpga_guidance(p)
r_neu = astar_search(p, cost)
r_van = astar_search(p, ones)


def render(problem, res):
    h, w = problem.height, problem.width
    canvas = np.stack([problem.obstacle_map] * 3, axis=-1).astype(float)
    canvas[res.frontier_map(h, w)] = [1.00, 0.93, 0.55]   # generated only
    canvas[res.closed_map(h, w)] = [0.20, 0.80, 0.00]     # expanded
    canvas[res.path_map(h, w)] = [1.00, 0.00, 0.00]       # path
    return canvas


fig, ax = plt.subplots(1, 3, figsize=(13, 5.2))
fig.subplots_adjust(top=0.80, bottom=0.06, wspace=0.12)

im = ax[0].imshow(cost, cmap="viridis")
ax[0].set_title("guidance map\ncomputed on the FPGA", fontsize=11, pad=12)
fig.colorbar(im, ax=ax[0], fraction=0.046, pad=0.04)

ax[1].imshow(render(p, r_van))
ax[1].set_title(f"vanilla A*\n{r_van.num_generated} generated / "
                f"{r_van.num_expansions} expanded", fontsize=11, pad=12)

ax[2].imshow(render(p, r_neu))
ax[2].set_title(f"Neural A* on hardware\n{r_neu.num_generated} generated / "
                f"{r_neu.num_expansions} expanded", fontsize=11, pad=12)

for a_ in ax:
    a_.axis("off")

from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(facecolor=(1.00, 0.93, 0.55), edgecolor="k", label="generated only"),
    Patch(facecolor=(0.20, 0.80, 0.00), edgecolor="k", label="expanded"),
    Patch(facecolor=(1.00, 0.00, 0.00), edgecolor="k", label="path"),
], loc="lower center", ncol=3, frameon=False, fontsize=10)

fig.suptitle("Neural A* on PYNQ-Z2 — 32x32 maze", fontsize=13, y=0.97)
fig.savefig(os.path.join(ROOT, "board_result.png"), dpi=150,
            bbox_inches="tight", facecolor="white")
print("saved board_result.png")
plt.show()

saved board_result.png


## 6. For the report

Everything above is measured on hardware. Fill in from the cells:

| | source |
|---|---|
| bitstream | tiny encoder, W4A8, 3,831 params |
| post-synthesis LUT / BRAM / DSP | `04_fast_track` section 7 |
| FPGA encode, measured | section 3 |
| FINN cycle estimate | 1.47 ms at 100 MHz |
| x86 CPU encode | 6.70 ms |
| expansion / generated reduction | section 4 |
| end-to-end per plan | section 4 |

Three things worth a sentence each:

**Measured latency exceeds the cycle estimate.** The estimate counts the
accelerator; the measurement includes DMA and a Python driver on a 650 MHz ARM
core. That difference is accelerator throughput versus system latency.

**The sigmoid runs on the host.** FINN left the final layer's INT24 accumulator
exposed rather than folding the sigmoid into thresholds, so section 2 fits the
recovering scale and offset. Standard practice — FINN classifiers return logits
the same way — and the quality of the fit is itself a correctness check.

**The ARM search is slower than the x86 one.** `01_baseline` measured 0.60 ms on
a desktop CPU; a Cortex-A9 will be several times that, which shifts the balance
between the two stages on this platform.

### Next build

`MAC_BUDGET = 54` in `04_fast_track` projects ~0.74 ms at roughly half the chip,
about 9× the CPU. One more synthesis gives a second measured point, and with the
sweep already in `sweep_results.json` that is enough to anchor the whole
frontier.